# Simulation of Rabi oscillations of a weakly coupled spin-1/2 pair

In [ ]:
# Imports
import matplotlib.pyplot as plt
import numpy as np
import scipy.constants as const
from qutip import (
    tensor, qeye, sigmax, sigmay, sigmaz,
    basis, mesolve, sigmam, Qobj, Bloch
)

from spinfinity import ESR, lind, CC, SpinSys, bath, hamil
from spinfinity import helperfunctions as hf

%load_ext autoreload
%autoreload 2

## Spin hamiltonian

In [ ]:
# Define the spin system
spin = SpinSys([0.5, 0.5])

spin.B = [0.0, 0.0, 0.5]  # T
spin.BTip = [1e-9, 0.0, 0.0]  # T
spin.gfactorvector = [2.0, 2.0]
spin.ReadoutSpin = 0
spin.tip = "f"
spin.T = 0.5  # K

# Exchange parameters expressed as energy / h in GHz
J_perp_GHz = 0.05
J_parallel_GHz = 0.1

# Convert to the meV units required by the Hamiltonian
exchange_meV = np.array([
    J_perp_GHz,
    J_perp_GHz,
    J_parallel_GHz,
]) * spin.GHztomeVConversion

hamil.assign_ExchangeInteraction(
    spin,
    [(0, 1, exchange_meV)],
)

# Do not add a separate dipolar interaction
spin.DipoleVector[:] = 0.0

In [ ]:
spin.BTip = [1e-9, 0.0, 0.02]

hamil.build_Hamiltonian(spin)
hamil.calc_EigEnergies(spin)
hamil.calc_EigStates(spin)

hamil.plot_EigenMatrix(spin, plot_type="p");
ESR.plot_FreqSweep(spin, freq_array=np.linspace(13.0, 15.0, 1000));

### Collapse operators

In [ ]:
# Tunneling Parameters
spin.V_DC = 0.0  # mV
spin.TipPolarization = [0, 0, 0.5]
spin.U = np.array([0.0, 0.0])
spin.G_st = 0.1e-9  # S
spin.G_ss = [1e-9, 1e-9]  # S
spin.G_tt = 1 # Allow for tip-tip scattering
spin.b0 = 0.0

# Relaxation
bath.calc_Rates(spin)
c_ops_T1 = lind.calc_collapseOperators(spin, recalcRates=True)


# Decoherence
T_phi = 400  # ns
c_op_Tphi_0 = lind.calc_dephasingOperator(spin, spin_idx=0, T_phi=T_phi)
c_op_Tphi_1 = lind.calc_dephasingOperator(spin, spin_idx=1, T_phi=T_phi)
spin.collapse_ops = c_ops_T1 + [c_op_Tphi_0] + [c_op_Tphi_1] 


### Rabi detuning map

In [ ]:
_ = lind.initialize_thermal(spin)
spin.RabiRate = 30e-3  # GHz

tau_array = np.linspace(0, 200, 500)  # ns
freq_array = np.linspace(14.4, 14.7, 400)
_, _, signal_map = CC.calc_RabiDetuningMap(spin, tau_array, freq_array)

# Remove tiny imaginary numerical noise; the plotted observable should be real
plot_data = np.real_if_close(signal_map, tol=1000)
plot_data = np.asarray(plot_data, dtype=float)

plt.figure(figsize=(8, 5))
extent = [
    tau_array[0], tau_array[-1],
    freq_array[0], freq_array[-1]
]

plt.imshow(
    plot_data,
    aspect='auto',
    origin='lower',
    extent=extent,
    interpolation='none',
    cmap='magma'
)

plt.xlabel(r'Pulse Width $\tau$ (ns)')
plt.ylabel(r'$f_{\mathrm{RF}}$ (GHz)')
plt.clim([-0.5, 0.5])
plt.colorbar(label=r'Final $\langle S_{z} \rangle$')
plt.tight_layout()
plt.show()